**MGMT298D: Science and Strategy of AI**

# Week 6: Building a Tiny LLM


In this notebook, we build a small GPT-style language model and train it on Yelp reviews.

The plan is to run three live experiments:

1. train on **10,000 reviews for 1 epoch**,
2. train on **500,000 reviews for 1 epoch**, and
3. train on **500,000 reviews for 5 epochs**.

After each experiment, we will inspect the immediate next-token predictions and generate a short continuation from the same prompt. The goal is to observe how more training data and more training time change the model's review-style predictions.


# Setup


In [ ]:
#@title Import libraries { display-mode: "form" }
import os
import logging
import warnings

# Keep Hugging Face public-dataset downloads quiet in Colab.
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", message=r".*HF_TOKEN.*", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*unauthenticated requests.*", category=UserWarning)
warnings.filterwarnings("ignore", module=r"huggingface_hub.*")

import numpy as np
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset, disable_progress_bar
from huggingface_hub.utils import disable_progress_bars

disable_progress_bar()
disable_progress_bars()

import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


---
# Load Yelp Reviews


The full Yelp Review Full training split is loaded once. Each experiment below chooses its own subset size from this shared text pool.


In [ ]:
dataset = load_dataset("yelp_review_full", split="train", token=False)

FULL_TRAIN_SIZE = len(dataset)
texts = dataset["text"]

print(f"Loaded {len(texts):,} Yelp reviews.")

print("\nRandom training examples:\n")
preview_rng = np.random.default_rng(seed=42)
preview_indices = preview_rng.choice(len(texts), size=min(50, len(texts)), replace=False)
for i, idx in enumerate(preview_indices, start=1):
    print(f"{i}. {texts[int(idx)]}\n")


---
# Experiment Settings


In [ ]:
#@title Shared settings { display-mode: "form" }
VOCAB_SIZE = 2000
SEQ_LEN = 64       # number of tokens the model can look back at
BATCH_SIZE = 32    # lower batch size keeps the large next-token output memory-safe
VOCAB_ADAPT_EXAMPLES = 100000
VALIDATION_FRACTION = 0.20
DEFAULT_PROMPT = "the fish tacos were"
GENERATED_TOKENS = 8

EMBED_DIM = 128    # size of each token vector
NUM_HEADS = 4      # number of attention heads per transformer block
FF_DIM = 256       # hidden size of the dense layers inside each transformer block
NUM_BLOCKS = 2     # number of transformer blocks below


In [ ]:
#@title Compact architecture overview { display-mode: "form" }
print(
    "Model overview\n"
    f"Token embeddings: vocabulary={VOCAB_SIZE:,}, embedding_dim={EMBED_DIM}\n"
    f"Position embeddings: context_length={SEQ_LEN}, embedding_dim={EMBED_DIM}\n"
    f"Transformer blocks: {NUM_BLOCKS}\n"
    f"Causal self-attention: {NUM_HEADS} heads, {EMBED_DIM // NUM_HEADS} dims per head\n"
    f"Dense layers: hidden_dim={FF_DIM}, output_dim={EMBED_DIM}\n"
    f"Next-token output layer: vocabulary={VOCAB_SIZE:,} logits\n"
    f"Training batch size: {BATCH_SIZE}"
)


---
# Data Pipeline


In [ ]:
#@title Build streaming language-model datasets { display-mode: "form" }
def lowercase_keep_punctuation(text):
    return tf.strings.lower(text)


def prepare_lm_datasets(num_reviews, seed=42):
    selected_texts = texts[:num_reviews]

    rng = np.random.default_rng(seed=seed)
    indices = rng.permutation(len(selected_texts))
    val_size = int(len(selected_texts) * VALIDATION_FRACTION)
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]

    train_texts = [selected_texts[int(i)] for i in train_idx]
    val_texts = [selected_texts[int(i)] for i in val_idx]

    vectorizer = layers.TextVectorization(
        max_tokens=VOCAB_SIZE,
        standardize=lowercase_keep_punctuation,
        output_sequence_length=SEQ_LEN + 1
    )

    adapt_size = min(VOCAB_ADAPT_EXAMPLES, len(train_texts))
    adapt_indices = rng.choice(len(train_texts), size=adapt_size, replace=False)
    adapt_texts = [train_texts[int(i)] for i in adapt_indices]

    print(f"Preparing {num_reviews:,} reviews...")
    print(f"Building vocabulary from {len(adapt_texts):,} sampled training reviews...")
    vectorizer.adapt(tf.data.Dataset.from_tensor_slices(adapt_texts).batch(1024))
    vocab = vectorizer.get_vocabulary()
    print(f"Vocabulary ready: {len(vocab):,} tokens.")

    def make_lm_dataset(text_batch):
        tokens = vectorizer(text_batch)
        return tokens[:, :-1], tokens[:, 1:]

    autotune = tf.data.AUTOTUNE
    shuffle_buffer = min(len(train_texts), 10000)

    train_ds = (
        tf.data.Dataset.from_tensor_slices(train_texts)
        .shuffle(shuffle_buffer, seed=seed, reshuffle_each_iteration=True)
        .batch(BATCH_SIZE)
        .map(make_lm_dataset, num_parallel_calls=autotune)
        .prefetch(autotune)
    )

    val_ds = (
        tf.data.Dataset.from_tensor_slices(val_texts)
        .batch(BATCH_SIZE)
        .map(make_lm_dataset, num_parallel_calls=autotune)
        .prefetch(autotune)
    )

    print(f"Training examples: {len(train_texts):,} | Validation examples: {len(val_texts):,}")
    return train_ds, val_ds, vectorizer, vocab


---
# Build a Tiny GPT-Style Language Model

This model is a small decoder-only transformer. It is much smaller than production LLMs, but it uses the same basic next-token prediction idea.

Architecture:

1. **Token embeddings** convert token IDs into vectors.
2. **Position embeddings** add information about where each token appears.
3. **Causal self-attention** lets each token use earlier tokens while preventing it from seeing future tokens.
4. **Dense layers** process the representation at each position.
5. **Residual connections and layer normalization** help stabilize training.
6. **The output layer** predicts logits over the vocabulary for the next token.

The model is built directly with Keras layers in each training experiment so the main pieces are visible.


---
# Generation Helpers

For each prompt, the widget shows:

1. the model's immediate next-token distribution, and
2. an 8-token generated continuation.


In [ ]:
#@title Define generation utilities + interactive widget { display-mode: "form" }
DEFAULT_TEMPERATURE = 1.1


def _prompt_tokens(prompt, vectorizer):
    tokens = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(tokens > 0)[0]
    if len(nonzero) == 0:
        return []
    return list(tokens[:nonzero[-1] + 1])


def next_token_probs(model, prompt, vectorizer):
    out = _prompt_tokens(prompt, vectorizer)
    if not out:
        return None

    padded = np.zeros(SEQ_LEN, dtype="int32")
    context = out[-SEQ_LEN:]
    padded[:len(context)] = context
    pred_pos = len(context) - 1

    logits = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
    probs = tf.nn.softmax(logits).numpy()
    probs = probs.copy()

    # Do not display or sample padding or unknown tokens.
    probs[0] = 0
    probs[1] = 0
    probs = probs / probs.sum()
    return probs


def plot_next_token_distribution(model, prompt, vectorizer, vocab, top_k=10):
    probs = next_token_probs(model, prompt, vectorizer)
    if probs is None:
        print("Type a prompt first.")
        return

    id_to_word = dict(enumerate(vocab))
    top_ids = np.argsort(probs)[-top_k:][::-1]
    words = [id_to_word[i] for i in top_ids]
    values = [probs[i] for i in top_ids]

    plt.figure(figsize=(6, 2.4))
    plt.bar(words, values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Probability")
    plt.title("Immediate next-token distribution")
    plt.tight_layout()
    plt.show()


def generate_text(model, prompt, vectorizer, vocab, length=GENERATED_TOKENS, temperature=DEFAULT_TEMPERATURE):
    id_to_word = dict(enumerate(vocab))
    out = _prompt_tokens(prompt, vectorizer)
    if not out:
        return "(type a prompt first)"

    prompt_len = len(out)
    temperature = max(float(temperature), 1e-6)

    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype="int32")
        context = out[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1

        logits = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        probs = tf.nn.softmax(logits / temperature).numpy()
        probs = probs.copy()
        probs[0] = 0
        probs[1] = 0
        probs = probs / probs.sum()

        next_id = np.random.choice(len(probs), p=probs)
        out.append(next_id)

    prompt_words = [id_to_word[t] for t in out[:prompt_len]]
    generated_words = [id_to_word[t] for t in out[prompt_len:]]
    return f"<b>{' '.join(prompt_words)}</b> {' '.join(generated_words)}"


def make_generator_widget(model, vectorizer, vocab, title="Try it yourself"):
    prompt_box = widgets.Text(
        value=DEFAULT_PROMPT,
        placeholder="Type a prompt...",
        description="Prompt:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "80px"}
    )
    temperature_slider = widgets.FloatSlider(
        value=DEFAULT_TEMPERATURE,
        min=0.2,
        max=1.5,
        step=0.1,
        description="Temp:",
        readout_format=".1f",
        layout=widgets.Layout(width="300px"),
        style={"description_width": "80px"}
    )

    button = widgets.Button(description="Generate", button_style="primary")

    chart_output = widgets.Output()
    text_output = widgets.Output(layout=widgets.Layout(min_height="50px", padding="8px"))

    def run_generation(_=None):
        with chart_output:
            chart_output.clear_output(wait=True)
            plot_next_token_distribution(model, prompt_box.value, vectorizer, vocab, top_k=10)

        with text_output:
            text_output.clear_output(wait=True)
            generated = generate_text(
                model,
                prompt_box.value,
                vectorizer,
                vocab,
                length=GENERATED_TOKENS,
                temperature=temperature_slider.value,
            )
            display(HTML(f"<div style='font-size:15px'>{generated}</div>"))

    button.on_click(run_generation)

    try:
        prompt_box.on_submit(run_generation)
    except Exception:
        pass

    display(HTML(f"<h4>{title}</h4>"))
    display(widgets.HBox([prompt_box, button]))
    display(temperature_slider)
    display(HTML("<b>Immediate next-token distribution</b>"))
    display(chart_output)
    display(HTML(f"<b>Generated continuation: next {GENERATED_TOKENS} tokens</b>"))
    display(text_output)

    run_generation()


---
# Experiment 1: 10,000 Reviews, 1 Epoch

This first run gives us a small-data baseline. We expect the model to learn common Yelp words quickly, but its generation may be generic or unstable.


In [ ]:
train_ds_10k, val_ds_10k, vectorizer_10k, vocab_10k = prepare_lm_datasets(10_000)

# Build the model manually: embeddings, causal attention, dense layers
inputs = layers.Input(shape=(SEQ_LEN,), name="tokens")

# -------------------------------------------------------------------
# 1. Token embeddings
# -------------------------------------------------------------------
token_embeddings = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    name="token_embedding"
)(inputs)

# -------------------------------------------------------------------
# 2. Position embeddings
# -------------------------------------------------------------------
positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
position_embeddings = layers.Embedding(
    input_dim=SEQ_LEN,
    output_dim=EMBED_DIM,
    name="position_embedding"
)(positions)

x = token_embeddings + position_embeddings

# -------------------------------------------------------------------
# 3. Transformer block 1
# -------------------------------------------------------------------
attn_1 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_1"
)(x, x, use_causal_mask=True)

x = layers.LayerNormalization(name="norm_1")(x + attn_1)

ff_1 = layers.Dense(FF_DIM, activation="relu", name="ff_1_dense_1")(x)
ff_1 = layers.Dense(EMBED_DIM, name="ff_1_dense_2")(ff_1)

x = layers.LayerNormalization(name="norm_2")(x + ff_1)

# -------------------------------------------------------------------
# 4. Transformer block 2
# -------------------------------------------------------------------
attn_2 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_2"
)(x, x, use_causal_mask=True)

x = layers.LayerNormalization(name="norm_3")(x + attn_2)

ff_2 = layers.Dense(FF_DIM, activation="relu", name="ff_2_dense_1")(x)
ff_2 = layers.Dense(EMBED_DIM, name="ff_2_dense_2")(ff_2)

x = layers.LayerNormalization(name="norm_4")(x + ff_2)

# -------------------------------------------------------------------
# 5. Next-token prediction head
# -------------------------------------------------------------------
outputs = layers.Dense(VOCAB_SIZE, name="next_token_logits")(x)

model_10k_1epoch = keras.Model(inputs, outputs)
model_10k_1epoch.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


In [ ]:
h_10k_1epoch = model_10k_1epoch.fit(train_ds_10k, epochs=1, validation_data=val_ds_10k)


In [ ]:
make_generator_widget(
    model_10k_1epoch,
    vectorizer_10k,
    vocab_10k,
    "Experiment 1: 10,000 reviews, 1 epoch"
)


---
# Experiment 2: 500,000 Reviews, 1 Epoch

This run keeps training time short but gives the model much more language variety. We can compare whether more data improves next-token predictions after only one pass.


In [ ]:
train_ds_500k, val_ds_500k, vectorizer_500k, vocab_500k = prepare_lm_datasets(500_000)

# Build the model manually: embeddings, causal attention, dense layers
inputs = layers.Input(shape=(SEQ_LEN,), name="tokens")

# -------------------------------------------------------------------
# 1. Token embeddings
# -------------------------------------------------------------------
token_embeddings = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    name="token_embedding"
)(inputs)

# -------------------------------------------------------------------
# 2. Position embeddings
# -------------------------------------------------------------------
positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
position_embeddings = layers.Embedding(
    input_dim=SEQ_LEN,
    output_dim=EMBED_DIM,
    name="position_embedding"
)(positions)

x = token_embeddings + position_embeddings

# -------------------------------------------------------------------
# 3. Transformer block 1
# -------------------------------------------------------------------
attn_1 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_1"
)(x, x, use_causal_mask=True)

x = layers.LayerNormalization(name="norm_1")(x + attn_1)

ff_1 = layers.Dense(FF_DIM, activation="relu", name="ff_1_dense_1")(x)
ff_1 = layers.Dense(EMBED_DIM, name="ff_1_dense_2")(ff_1)

x = layers.LayerNormalization(name="norm_2")(x + ff_1)

# -------------------------------------------------------------------
# 4. Transformer block 2
# -------------------------------------------------------------------
attn_2 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_2"
)(x, x, use_causal_mask=True)

x = layers.LayerNormalization(name="norm_3")(x + attn_2)

ff_2 = layers.Dense(FF_DIM, activation="relu", name="ff_2_dense_1")(x)
ff_2 = layers.Dense(EMBED_DIM, name="ff_2_dense_2")(ff_2)

x = layers.LayerNormalization(name="norm_4")(x + ff_2)

# -------------------------------------------------------------------
# 5. Next-token prediction head
# -------------------------------------------------------------------
outputs = layers.Dense(VOCAB_SIZE, name="next_token_logits")(x)

model_500k_1epoch = keras.Model(inputs, outputs)
model_500k_1epoch.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


In [ ]:
h_500k_1epoch = model_500k_1epoch.fit(train_ds_500k, epochs=1, validation_data=val_ds_500k)


In [ ]:
make_generator_widget(
    model_500k_1epoch,
    vectorizer_500k,
    vocab_500k,
    "Experiment 2: 500,000 reviews, 1 epoch"
)


---
# Experiment 3: 500,000 Reviews, 5 Epochs

This final run uses the larger dataset and gives the same architecture more time to learn. We expect better review-like phrasing, stronger next-token distributions, and more coherent short continuations.


In [ ]:
# Continue training the 500k-review model for 4 more epochs.
# Together with Experiment 2, this gives 5 total epochs on the same data.
h_500k_extra_epochs = model_500k_1epoch.fit(train_ds_500k, epochs=4, validation_data=val_ds_500k)
model_500k_5epochs = model_500k_1epoch


In [ ]:
make_generator_widget(
    model_500k_5epochs,
    vectorizer_500k,
    vocab_500k,
    "Experiment 3: 500,000 reviews, 5 epochs"
)


---
# Compare Training Curves


In [ ]:
def plot_history_values(losses, val_losses, label, ax):
    epochs = range(1, len(losses) + 1)
    ax.plot(epochs, losses, "o-", label=f"{label} train")
    ax.plot(epochs, val_losses, "o--", label=f"{label} val")

loss_500k_5epochs = list(h_500k_1epoch.history["loss"]) + list(h_500k_extra_epochs.history["loss"])
val_500k_5epochs = list(h_500k_1epoch.history["val_loss"]) + list(h_500k_extra_epochs.history["val_loss"])

fig, ax = plt.subplots(figsize=(8, 4))
plot_history_values(h_10k_1epoch.history["loss"], h_10k_1epoch.history["val_loss"], "10k / 1 epoch", ax)
plot_history_values(h_500k_1epoch.history["loss"], h_500k_1epoch.history["val_loss"], "500k / 1 epoch", ax)
plot_history_values(loss_500k_5epochs, val_500k_5epochs, "500k / 5 epochs", ax)
ax.set(xlabel="Epoch", ylabel="Loss", title="Training and Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()
